## Load dataset

In [1]:
import numpy as np
import matplotlib
import pandas as pd
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats, integrate, optimize

df = pd.read_excel("MASD2024_data.xlsx")

In [2]:
print("Shape:", df.shape)

display(df.head())

#df.info()



Shape: (34, 6)


,Have you ever taken a mathematics and/or statistics course before?,How much do you expect to like this course?,What grade do you expect to get in the course?,Please rate your level of experience with programming,The thought of being enrolled in a mathematics/statistics course makes me nervous.,What type of high-school did you attend?
0,Yes,4,4,6,5.0,HTX
1,Yes,5,10,7,1.0,HTX
2,No,3,4,3,4.0,STX
3,No,2,4,5,6.0,STX
4,Yes,3,4,4,6.0,HTX


## Probability mass function of expected grades

In [3]:
pmf = df["What grade do you expect to get in the course?"].value_counts(normalize=True).sort_index()

pmf

What grade do you expect to get in the course?
0     0.029412
2     0.058824
4     0.382353
7     0.264706
10    0.235294
12    0.029412
Name: proportion, dtype: float64

In [4]:
import matplotlib.pyplot as plt

pmf.plot(kind="bar")

plt.xlabel("Expected grade")
plt.ylabel("Probability")
plt.title("PMF of expected grades")

plt.show()

C:\Users\bruge\AppData\Local\Temp\ipykernel_15828\3982518900.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
import numpy as np


# Calculate the expected value of the PMF, by multiplying each grade by its probability and summing the results
expected_value = np.sum(pmf.index * pmf)

print(expected_value)

6.205882352941177


In [6]:
ex2 = np.sum((pmf.index**2) * pmf.values)

# Calculate the variance of the PMF, by subtracting the square of the expected value from the expected value of the squared grades
variance = ex2 - expected_value**2

print(variance)


8.575259515570927


## Predicting nervousness based on high school type

## Modelling expected grades with a cumulative distribution function

In [ ]:
# cumulative distribution function (CDF)
def F(x):
    x = np.asarray(x, dtype=float)
    return np.piecewise(
        x,
        [x < -3,
         (x >= -3) & (x < 2),
         (x >= 2) & (x < 7),
         (x >= 7) & (x < 12),
         x >= 12],
        [0,
         lambda t: 0.0588 * (t + 3),
         lambda t: 0.294 + 0.1176 * (t - 2),
         lambda t: 0.882 + 0.0235 * (t - 7),
         1])

In [23]:
# a)
print(f"P(X < 7) = F(7) = {F(7):.4f}")

P(X < 7) = F(7) = 0.8820


In [24]:
# calculate median
med = optimize.brentq(lambda x: F(x) - 0.5, -3, 12)
print(f"Median = {med:.4f}")

Median = 3.7517


In [ ]:
# c)
def f(x):
    x = np.asarray(x, dtype=float)
    return np.piecewise(
        x,
        [(x >= -3) & (x < 2),
         (x >= 2) & (x < 7),
         (x >= 7) & (x < 12)],
        [0.0588, 0.1176, 0.0235, 0.0])

f()
 
print("f(x) = 0.0588 på [-3,2), 0.1176 på [2,7), 0.0235 på [7,12), 0 ellers")

c) PDF: f(x) = 0.0588 på [-3,2), 0.1176 på [2,7), 0.0235 på [7,12), 0 ellers


In [11]:
pieces = [(-3, 2, 0.0588), (2, 7, 0.1176), (7, 12, 0.0235)]
total_mass = sum(p * (b - a) for a, b, p in pieces)
EX = sum(p * (b**2 - a**2) / 2 for a, b, p in pieces)
EX2 = sum(p * (b**3 - a**3) / 3 for a, b, p in pieces)
VarX = EX2 - EX**2
print(f"   Samlet sandsynlighedsmasse = {total_mass:.4f} (bør være ~1)")
print(f"   E[X]   = {EX:.4f}")
print(f"   E[X^2] = {EX2:.4f}")
print(f"   Var(X) = {VarX:.4f}, std = {np.sqrt(VarX):.4f}")

   Samlet sandsynlighedsmasse = 0.9995 (bør være ~1)
   E[X]   = 3.6153
   E[X^2] = 24.6672
   Var(X) = 11.5971, std = 3.4055


In [12]:
EX_num = sum(integrate.quad(lambda x: x * f(x), a, b)[0] for a, b, _ in pieces)
EX2_num = sum(integrate.quad(lambda x: x**2 * f(x), a, b)[0] for a, b, _ in pieces)
print(f"   Kontrol (numerisk): E[X]={EX_num:.4f}, Var={EX2_num - EX_num**2:.4f}")

   Kontrol (numerisk): E[X]=3.6153, Var=11.5971


In [ ]:
# d)
xs = np.linspace(-5, 14, 1000)
plt.

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(xs, F(xs), color="b")
ax[0].set_title("CDF for forventede karakterer")
ax[0].set_xlabel("x")
ax[0].set_ylabel("F(x)")
ax[0].grid(alpha=0.3)

ax[1].plot(xs, f(xs), color="r")
ax[1].set_title("PDF (afledt af CDF)")
ax[1].set_xlabel("x")
ax[1].set_ylabel("f(x)")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


C:\Users\bruge\AppData\Local\Temp\ipykernel_15828\4091289329.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
print("d) Sandsynlighedsmasse pr. interval:")
for a, b, p in pieces:
    print(f"   [{a:>3}, {b:>2}): masse = {p*(b-a):.4f}, tæthed = {p}")
print("   -> Tætheden er højest på [2,7), som rummer ca. 58.8% af massen.")

d) Sandsynlighedsmasse pr. interval:
   [ -3,  2): masse = 0.2940, tæthed = 0.0588
   [  2,  7): masse = 0.5880, tæthed = 0.1176
   [  7, 12): masse = 0.1175, tæthed = 0.0235
   -> Tætheden er højest på [2,7), som rummer ca. 58.8% af massen.


## Normal distribution

In [15]:
mu, sigma = 3.71, 1.25
X = stats.norm(loc=mu, scale=sigma)

In [16]:
# a)
z6 = (6 - mu) / sigma
print(f"a) z = {z6:.4f}, P(X > 6) = {X.sf(6):.4f}")

a) z = 1.8320, P(X > 6) = 0.0335


In [17]:
# b)
z2, z4 = (2 - mu) / sigma, (4 - mu) / sigma
print(f"b) z2 = {z2:.4f}, z4 = {z4:.4f}, P(2 <= X <= 4) = {X.cdf(4) - X.cdf(2):.4f}")

b) z2 = -1.3680, z4 = 0.2320, P(2 <= X <= 4) = 0.5061


In [18]:
# c)
p_c = X.cdf(2) / X.cdf(4)
print(f"c) P(X<=2) = {X.cdf(2):.4f}, P(X<4) = {X.cdf(4):.4f}, "
      f"P(X<=2 | X<4) = {p_c:.4f}")

c) P(X<=2) = 0.0857, P(X<4) = 0.5917, P(X<=2 | X<4) = 0.1448


In [19]:
# d)
z95 = stats.norm.ppf(0.95)
x_d = mu + sigma * z95
print(f"d) z_0.95 = {z95:.4f}, x = {x_d:.4f} (kontrol: {X.isf(0.05):.4f})")

d) z_0.95 = 1.6449, x = 5.7661 (kontrol: 5.7661)


In [21]:
xs = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 500)

plt.figure(figsize=(7, 4))
plt.plot(xs, X.pdf(xs), color="b")
plt.fill_between(xs, X.pdf(xs), where=(xs >= 2) & (xs <= 4), alpha=0.3, label="2 ≤ X ≤ 4")
plt.fill_between(xs, X.pdf(xs), where=(xs > 6), alpha=0.3, color="r", label="X > 6")
plt.axvline(x_d, color="k", ls="--", label=f"x = {x_d:.2f} (5% øvre hale)")
plt.title("N(3.71, 1.25²)")
plt.xlabel("x")
plt.ylabel("Tæthed")
plt.legend()
plt.tight_layout()
plt.show()

C:\Users\bruge\AppData\Local\Temp\ipykernel_15828\2760489977.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
